In [ ]:
%%configure -f
{
    "defaultLakehouse": { "name": "SilverLakehouse" }
}

# Silver - SIS (SQL)  ·  Issue #15
Reads the **Bronze SIS mirror** (Azure SQL mirrored into OneLake) and writes clean,
conformed **Silver** dimensions and facts into `SilverLakehouse` under the **`dbo`** schema.
**What it does**
1. Attaches `SilverLakehouse` as the default lakehouse (by name - no tenant GUIDs committed).
2. Resolves the SIS mirror item at runtime and reads its 11 tables via OneLake (by item id,
   because the mirror's display name contains a space that breaks name-based ABFS paths).
3. Renames columns to `snake_case`, trims strings, enforces the primary key (non-null + de-duplicated).
4. Writes each as a managed Delta table `dbo.<table>` (overwrite = idempotent / re-runnable).
**Silver output** (answers demo questions 1-3, 6): `dbo.department`, `dbo.term`,
`dbo.building`, `dbo.classroom`, `dbo.program`, `dbo.professor`, `dbo.course`,
`dbo.student`, `dbo.course_section`, `dbo.enrollment`, `dbo.student_term_status`.
> The pre-baked `CourseReviewSignal` table is intentionally **skipped** here - the
> `Silver - Course Reviews (AI)` notebook re-derives that signal from the raw review text.

In [ ]:
import re
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Resolve the workspace at runtime so this notebook is tenant-agnostic (no committed GUIDs).
try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
except Exception:
    WORKSPACE_ID = spark.conf.get("trident.workspace.id")

ONELAKE = "onelake.dfs.fabric.microsoft.com"
SILVER_SCHEMA = "dbo"

def resolve_item_id(display_name: str, item_type: str) -> str:
    """Look up a workspace item's id by display name (keeps ABFS paths GUID-free in Git)."""
    tok = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items?type={item_type}"
    r = requests.get(url, headers={"Authorization": f"Bearer {tok}"}, timeout=60)
    r.raise_for_status()
    for it in r.json()["value"]:
        if it["displayName"] == display_name:
            return it["id"]
    raise ValueError(f"{item_type} '{display_name}' not found in workspace {WORKSPACE_ID}")

def item_path(item_id: str, *segments: str) -> str:
    """Build an ABFS OneLake path for an item id in the current workspace."""
    base = f"abfss://{WORKSPACE_ID}@{ONELAKE}/{item_id}"
    return base + ("/" + "/".join(segments) if segments else "")

# Bronze source = SIS mirror (resolved by id); Silver target = attached default lakehouse.
SIS_MIRROR_ID = resolve_item_id("SIS Mirror", "MirroredDatabase")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print(f"Workspace  : {WORKSPACE_ID}")
print(f"SIS mirror : {SIS_MIRROR_ID}")
print(f"Target     : default lakehouse SilverLakehouse, schema '{SILVER_SCHEMA}'")

In [ ]:
def to_snake(name: str) -> str:
    """CamelCase / PascalCase -> snake_case (StudentID -> student_id, LmsLogins -> lms_logins)."""
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    s = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", "_", s)
    return s.lower()

def load_clean_write(source_table: str, silver_table: str, pk: str):
    """Read one Bronze SIS table, conform it, and write it to dbo.<silver_table>."""
    src = item_path(SIS_MIRROR_ID, "Tables", "dbo", source_table)
    df = spark.read.format("delta").load(src)

    # rename all columns to snake_case
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake(c))
    pk_sc = to_snake(pk)

    # trim whitespace on string columns
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))

    # enforce the primary key: drop null PKs, keep one row per key
    df = df.where(F.col(pk_sc).isNotNull()).dropDuplicates([pk_sc])

    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.{silver_table}"))
    n = spark.table(f"{SILVER_SCHEMA}.{silver_table}").count()
    print(f"  dbo.{silver_table:<20} <- dbo.{source_table:<18} rows={n}")
    return silver_table, n

In [ ]:
# (source table, silver table, primary key)
SIS_TABLES = [
    # dimensions
    ("Department",        "department",           "DeptID"),
    ("Term",              "term",                 "TermID"),
    ("Building",          "building",             "BldID"),
    ("Classroom",         "classroom",            "RoomID"),
    ("Program",           "program",              "ProgramID"),
    ("Professor",         "professor",            "ProfID"),
    ("Course",            "course",               "CrsID"),
    ("Student",           "student",              "StudentID"),
    # facts
    ("CourseSection",     "course_section",       "SectionID"),
    ("Enrollment",        "enrollment",           "EnrollmentID"),
    ("StudentTermStatus", "student_term_status",  "StatusID"),
]

print("Writing Silver (dbo schema):")
results = [load_clean_write(src, dst, pk) for src, dst, pk in SIS_TABLES]
print(f"\nDone. {len(results)} Silver tables written to {SILVER_SCHEMA}.*")

In [ ]:
# Quick peek: student demographics, to confirm the Silver write.
display(spark.table(f"{SILVER_SCHEMA}.student").limit(5))